In [ ]:

#IMPORT THE LIBRARIES
import pandas as pd
import numpy as np
import os


# librosa is a Python library for analyzing audio and music. It can be used to extract the data from the audio files we will see it later.
import librosa
import librosa.display
import seaborn as sns
import matplotlib.pyplot as plt

import cupy as cp
from tqdm import tqdm
from joblib import Parallel, delayed
import timeit

from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
# prompt: read data_path = pd.read_csv('/content/drive/MyDrive/pfe/data_path_contact.csv')

import pandas as pd
data_path = pd.read_csv('/content/drive/MyDrive/pfe-data/data_path_contact.csv')

data_path.head()

,Emotions,Path
0,neutral,/content/drive/MyDrive/pfe/Ravdess/audio_speec...
1,surprise,/content/drive/MyDrive/pfe/Ravdess/audio_speec...
2,sad,/content/drive/MyDrive/pfe/Ravdess/audio_speec...
3,happy,/content/drive/MyDrive/pfe/Ravdess/audio_speec...
4,disgust,/content/drive/MyDrive/pfe/Ravdess/audio_speec...


In [ ]:
import librosa
import cupy as cp
import numpy as np
import pandas as pd
import os
from joblib import Parallel, delayed
from tqdm import tqdm

# GPU-accelerated augmentation functions
def noise(data, noise_level=0.035):
    noise_amp = noise_level * cp.random.uniform() * cp.amax(data)
    return cp.clip(data + noise_amp * cp.random.normal(size=data.shape[0]), -1.0, 1.0)

def stretch(data, rate=None):
    rate = rate if rate else float(cp.random.uniform(0.8, 1.2))  # Convert to Python float
    # Use a GPU-accelerated library for time stretching if possible
    return cp.asarray(librosa.effects.time_stretch(y=cp.asnumpy(data), rate=rate))  # Back to GPU

def shift(data):
    shift_range = int(cp.random.uniform(low=-1000, high=1000))
    return cp.roll(data, shift_range)

def pitch(data, sampling_rate, pitch_factor=None):
    pitch_factor = pitch_factor if pitch_factor else float(cp.random.uniform(-3, 3))  # Convert to Python float
    return cp.asarray(librosa.effects.pitch_shift(y=cp.asnumpy(data), sr=sampling_rate, n_steps=pitch_factor))

def volume_scale(data, scale_range=(0.5, 1.5)):
    scale = cp.random.uniform(low=scale_range[0], high=scale_range[1])
    return cp.clip(data * scale, -1.0, 1.0)

def higher_speed(data, speed_factor=1.25):
    return cp.asarray(librosa.effects.time_stretch(y=cp.asnumpy(data), rate=speed_factor))

def lower_speed(data, speed_factor=0.75):
    return cp.asarray(librosa.effects.time_stretch(y=cp.asnumpy(data), rate=speed_factor))

# Feature extraction function with error handling and redundant RMS calculation removed
def extract_features(data, sample_rate):
    try:
        data = cp.asnumpy(data)  # Convert to NumPy for librosa functions

        # Extract features
        zcr = np.mean(librosa.feature.zero_crossing_rate(y=data).T, axis=0)
        stft = np.abs(librosa.stft(data))
        chroma_stft = np.mean(librosa.feature.chroma_stft(S=stft, sr=sample_rate).T, axis=0)
        rms = np.mean(librosa.feature.rms(y=data).T, axis=0)

        # Mel Spectrogram
        mel_spectrogram = librosa.feature.melspectrogram(y=data, sr=sample_rate)
        mel = np.mean(mel_spectrogram.T, axis=0)

        # MFCC (Mel-Frequency Cepstral Coefficients)
        mfcc = librosa.feature.mfcc(y=data, sr=sample_rate, n_mfcc=40)
        mfcc = np.mean(mfcc.T, axis=0)

        # Spectral Contrast and Tonnetz
        spectral_contrast = np.mean(librosa.feature.spectral_contrast(y=data, sr=sample_rate).T, axis=0)
        tonnetz = np.mean(librosa.feature.tonnetz(y=data, sr=sample_rate).T, axis=0)

        # Combine all features
        features = np.concatenate([zcr, chroma_stft, rms, mel, mfcc, spectral_contrast, tonnetz])

        print(f"Extracted features for sample rate {sample_rate}: {features.shape}")
        return features
    except Exception as e:
        print(f"Error extracting features: {e}")
        return None

# Improved feature extraction with error handling
def get_features(path, duration=2.5, offset=0.6):
    try:
        # Load audio file
        data, sample_rate = librosa.load(path, duration=duration, offset=offset)
        print(f"Loaded {path} successfully. Sample rate: {sample_rate}, Duration: {len(data)/sample_rate:.2f}s")
        data = data / np.max(np.abs(data))  # Normalize audio
        data_gpu = cp.asarray(data)  # Move data to GPU

        features = []

        # Define augmentations
        augmentations = [
            lambda x: x,  # Original
            noise,
            lambda x: pitch(stretch(x), sample_rate),
            shift,
            volume_scale,
            higher_speed,
            lower_speed
        ]

        # Apply augmentations and extract features
        for augment in augmentations:
            augmented_data = augment(data_gpu)
            feature = extract_features(augmented_data, sample_rate)
            if feature is not None:
                features.append(feature)

        # Debug: Print the shape of the features
        print(f"Features shape for {path}: {np.vstack(features).shape}")
        return np.vstack(features)
    except Exception as e:
        print(f"Error processing file {path}: {e}")
        return None

# Save extracted features to CSV
def save_features_to_disk(X, Y, filename='/content/drive/MyDrive/pfe/features-gpu-final.csv'):
    if not X:
        print("⚠️ No features to save. X is empty.")
        return

    # Create DataFrame
    df = pd.DataFrame(X)
    df['emotion'] = Y

    # Debug: Print the first few rows of the DataFrame
    print("Data to be saved:")
    print(df.head())

    # Save to CSV
    df.to_csv(filename, mode='a', header=not os.path.exists(filename), index=False)
    print(f"✅ Features saved to {filename}")

def process_data(data_path):
    X, Y = [], []

    # Process all files in parallel
    results = Parallel(n_jobs=-1)(  # Use all available cores
        delayed(get_features)(path)
        for path in tqdm(data_path.Path, total=len(data_path))
    )

    # Collect features and labels
    for features, emotion in zip(results, data_path.Emotions):
        if features is not None:
            X.extend(features)
            Y.extend([emotion] * len(features))
        else:
            print(f"⚠️ No features for file with emotion: {emotion}")

    # Debug: Print the total number of features and labels
    print(f"Total features collected: {len(X)}")
    print(f"Total labels collected: {len(Y)}")

    if not X:
        print("⚠️ No features collected. Check your data and feature extraction.")
        return

    # Save features to disk
    save_features_to_disk(X, Y)


In [ ]:
start = timeit.default_timer()
process_data(data_path)
stop = timeit.default_timer()
print('✅ Processing Done')
print('⏱️ Total Time: ', stop - start)

100%|██████████| 12162/12162 [3:19:34<00:00,  1.02it/s]


⚠️ No features for file with emotion: sad
Total features collected: 85127
Total labels collected: 85127
Data to be saved:
          0         1         2         3         4         5         6  \
0  0.182970  0.515045  0.529754  0.608991  0.683982  0.659834  0.569502   
1  0.241405  0.621877  0.630175  0.712308  0.773238  0.751281  0.658045   
2  0.153474  0.582200  0.526100  0.517605  0.591487  0.645550  0.633153   
3  0.182350  0.514367  0.532612  0.614993  0.684242  0.655253  0.567625   
4  0.182970  0.515035  0.529736  0.608977  0.683979  0.659833  0.569499   

          7         8         9  ...        186        187        188  \
0  0.550935  0.567459  0.547486  ...  19.006580  18.431348  43.920981   
1  0.594538  0.607712  0.588229  ...  15.786403  14.451687  13.765523   
2  0.590656  0.572146  0.561437  ...  19.991799  20.171464  43.320477   
3  0.553018  0.568303  0.548509  ...  18.862923  18.628619  43.987869   
4  0.550931  0.567459  0.547487  ...  19.020104  18.427830  42

In [ ]:

Emotions = pd.read_csv('/content/drive/MyDrive/pfe/features-gpu-final.csv')

# ... (rest of the code from the provided file)
Emotions.head()

,0,1,2,3,4,5,6,7,8,9,...,186,187,188,189,190,191,192,193,194,emotion
0,0.182970,0.515045,0.529754,0.608991,0.683982,0.659834,0.569502,0.550935,0.567459,0.547486,...,19.006580,18.431348,43.920981,-0.031129,0.035347,0.030434,-0.033044,0.008114,0.014114,neutral
1,0.241405,0.621877,0.630175,0.712308,0.773238,0.751281,0.658045,0.594538,0.607712,0.588229,...,15.786403,14.451687,13.765523,-0.021032,0.033963,0.028377,-0.030013,-0.000709,0.008721,neutral
2,0.153474,0.582200,0.526100,0.517605,0.591487,0.645550,0.633153,0.590656,0.572146,0.561437,...,19.991799,20.171464,43.320477,0.013674,-0.055683,0.050083,0.040258,0.013969,-0.008002,neutral
3,0.182350,0.514367,0.532612,0.614993,0.684242,0.655253,0.567625,0.553018,0.568303,0.548509,...,18.862923,18.628619,43.987869,-0.039866,0.038383,0.032842,-0.029110,0.008174,0.012371,neutral
4,0.182970,0.515035,0.529736,0.608977,0.683979,0.659833,0.569499,0.550931,0.567459,0.547487,...,19.020104,18.427830,42.621235,-0.031130,0.035355,0.030451,-0.033064,0.008106,0.014115,neutral
